In [11]:
# Numerical integration with QuadGK
# iteration with map function
# Mac Studio
# 2025-03-27

import Pkg

c = ["QuadGK", "BenchmarkTools", "Metal", "Chain"]
Pkg.add(c)

using QuadGK, BenchmarkTools, Metal, Chain

f = function (x)
    ν = 1.1915889f0
    μ = 0.1697106f0
    σ = 2.2166964f0
    t = (x - μ) / σ
    return (1 + t^2 / ν)^(-(ν + 1) / 2)
end

function main()
    num_points = 200 * 200
    num_edges = 176
    
    α = Float32(0.5)
    div_n = 
    @chain Metal.current_device() begin
        _.maxBufferLength
        floor(α * (_ / (sizeof(Float32) * num_points * num_edges)))
        convert(Int, _)
    end
    
    x0 = Array{Float32}(rand(num_points, div_n))
    x1 = Array{Float32}(rand(num_points, div_n))
    
    map(integral -> integral[1], quadgk.(f, x0[:, :], x1[:, :]))
    
end

@benchmark main()


   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


BenchmarkTools.Trial: 1 sample with 1 evaluation.
 Single result which took 7.064 s (1.69% GC) to evaluate,
 with a memory estimate of 1.12 GiB, over 36 allocations.

In [ ]:
# Numerical integration with Metal (trapezoidal rule)
# Mac Studio
# 2025-03-30

using Pkg
Pkg.add("Distributions")

using Distributions, Metal, Chain, Random, BenchmarkTools

function gpu_rand_float32(m::Int, n::Int, x::Float32, y::Float32)
    dist = Uniform(x, y)
    random_array_gpu = MtlArray{Float32}(undef, m, n)
    rand!(dist, random_array_gpu)

    return random_array_gpu

end

# t-distribution probability density function
# Attenuation function
function td(x)
    ν = 1.1915889f0
    μ = 0.1697106f0
    σ = 2.2166964f0
    t = (x - μ) / σ
    return (1 + t^2 / ν)^(-(ν + 1) / 2)
end

# Kernel function(trapezoidal rule)
function kernel_func(x0, y, steps, Δ)
    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y
    
    a = x0[i, j]
    
    integral = 0.0f0
    for k in 0:steps[i, j]-1
        x1 = a + k * Δ
        x2 = a + (k + 1) * Δ
        integral += (td(x1) + td(x2)) / 2 * Δ
    end
    
    y[i, j] = integral
    
    return nothing
end

function main()
    num_points = 200 * 200
    num_edges = 176

    α = Float32(0.5)
    div_n = 
    @chain Metal.current_device() begin
        _.maxBufferLength
        floor(α * (_ / (sizeof(Float32) * num_points * num_edges)))
        convert(Int, _)
    end
    
    dims = (num_points, div_n)
    dist_gpu = gpu_rand_float32(num_points, div_n, 0.0f0, 50.0f0)

    adj_dist_gpu = MtlArray(dist_gpu)
    start_p = Metal.zeros(Float32, num_points, div_n)
    
    # steps = Array{Int32}(undef, dims)
    # # Δ = 0.0009f0  # 積分の分割幅 # NG
    # # Δ = 0.001f0  # 積分の分割幅 # OK ?
    # # Δ = 0.005f0  # 積分の分割幅 # OKの時もある
    Δ = 0.00625f0  # 積分の分割幅 # OK 繰り返すとNGの時もある
    # # # Δ = 0.0075f0  # 積分の分割幅 # OK
    # # Δ = 0.01f0  # 積分の分割幅 # OK
    # for j in 1:div_num
    #     for i in 1:point_num
    #         steps[i, j] = round(Int32, dist[i, j] / Δ)
    #     end
    # end
    # steps_gpu = MtlArray(steps)

    steps_gpu = MtlArray{Int32}(undef, dims)
    @. steps_gpu = unsafe_trunc(Int32, dist_gpu / Δ)

    # Threadgroup size configuration
    threads_per_group = (32, 32)
    num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))

    # Kernel function execution
    @metal threads = threads_per_group groups = num_groups kernel_func(start_p, adj_dist_gpu, steps_gpu, Δ)
   adj_dist = Array(adj_dist_gpu)

end

@benchmark main()

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


BenchmarkTools.Trial: 2 samples with 1 evaluation per sample.
 Range (min … max):  4.443 s … 9.214 s  ┊ GC (min … max): 0.30% … 0.27%
 Time  (median):     6.828 s            ┊ GC (median):    0.28%
 Time  (mean ± σ):   6.828 s ± 3.374 s  ┊ GC (mean ± σ):  0.28% ± 0.02%

  █                                                     █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  4.44 s        Histogram: frequency by time       9.21 s <

 Memory estimate: 314.06 MiB, allocs estimate: 992.

In [ ]:
# t-distribution cumulative distribution function
# Mac Studio
# 2025-03-30

using Pkg
Pkg.add("Distributions")
using Distributions, Chain, Metal, Random, BenchmarkTools

function rand_float32(m::Int, n::Int, x::Float32, y::Float32)
    dist = Uniform(x, y)
    random_array = Array{Float32}(undef, m, n)
    rand!(dist, random_array)

    return random_array

end

function main()
    point_num = 200 * 200
    edge_num = 176

    α = Float32(0.5)
    div_n = 
    @chain Metal.current_device() begin
        _.maxBufferLength
        floor(α * (_ / (sizeof(Float32) * point_num * edge_num)))
        convert(Int, _)
    end

    ν = 1.191588

    d = TDist(ν)
    adj_dist = rand_float32(point_num, div_n, 0.0f0, 50.0f0)
    prob = similar(adj_dist)
    @. prob = cdf(d, adj_dist) - 0.5f0 # cdf(d, 0) = 0.5

end

@benchmark main()

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


BenchmarkTools.Trial: 3 samples with 1 evaluation per sample.
 Range (min … max):  1.764 s …   1.792 s  ┊ GC (min … max): 0.14% … 1.16%
 Time  (median):     1.768 s              ┊ GC (median):    0.14%
 Time  (mean ± σ):   1.775 s ± 15.476 ms  ┊ GC (mean ± σ):  0.45% ± 0.62%

  █        █                                              █  
  █▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  1.76 s         Histogram: frequency by time        1.79 s <

 Memory estimate: 209.35 MiB, allocs estimate: 7.

In [ ]:
# Threadgroupのサイズを設定
# t分布累積分布関数

using Pkg
Pkg.add("Distributions")

using Distributions
using Metal

# t分布の確率密度関数 (自由度ν=1.1915889, 位置母数μ=0.1697106, 尺度母数σ=2.2166964)
# 低減回帰関数

# カーネル関数
# t分布累積分布関数
function kernel_func(x0, y)

    ν = 1.1915889f0
    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y
        
    y[i, j] = cdf(TDist(ν), x0[i, j])
    
    return nothing
end

function main()
    point_num = 300 * 300
    # div_num = 2000
    # point_num = 500
    # div_num = 750 #OK
    # div_num = 1000 #OK
    div_num = 2000 #OK
    dims = (point_num, div_num)
    dist = Array{Float32}(undef, dims)
    conv_dist = similar(dist)

    for j in 1:div_num
        for i in 1:point_num
            # dist[i, j] = 0.1f0 * i
            dist[i, j] = 50.0f0 / 90000.0f0 * (i - 1)
        end
    end

    dist_gpu = MtlArray(dist)
    conv_dist_gpu = MtlArray(conv_dist)

    # Threadgroupのサイズを設定
    threads_per_group = (32, 32)
    num_groups = (ceil(Int, point_num / threads_per_group[1]), ceil(Int, div_num / threads_per_group[2]))

    # カーネル関数を実行
    @metal threads = threads_per_group groups = num_groups kernel_func(dist_gpu, conv_dist_gpu)
    conv_dist = Array(conv_dist_gpu)

end

main()